In [1]:
import os
import pandas as pd
import cv2
import numpy as np
from tqdm import tqdm
import time
import json

In [2]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles
from mediapipe.tasks.python import vision

In [3]:
# Constants

NUM_ROWS = 8192

DATASET_PATH = "../../../dataset"
IMAGES_A_PATH = os.path.join(DATASET_PATH, 'images_A')
IMAGES_B_PATH = os.path.join(DATASET_PATH, 'images_B')
METADATA_PATH = os.path.join(DATASET_PATH, 'metadata_A')

DATAFRAME_PATH = "../../eval"
SAVE_PATH = os.path.join(DATAFRAME_PATH, "pipeline_results.csv")

MODEL_PATH = "../../models/pose_landmarker_full.task"

FOCAL_LENGTH_MM = 35.0
SENSOR_WIDTH_MM = 36.0
IMAGE_HEIGHT_PX = 1080
IMAGE_WIDTH_PX = 1920
BASELINE_M = 0.1

In [4]:
# Load Dataframe

df = pd.read_csv(SAVE_PATH, dtype={'id': str})
df.head(6)

,id,actor,pose,cam_pitch,gt_dist,gt_ang_sep,gt_d_yaw,gt_d_pitch,gt_sw_x,gt_sw_y,...,m2st_dist,m2st_ang_sep,m2st_d_yaw,m2st_d_pitch,m2st_sw_x,m2st_sw_y,m2st_sw_z,m2st_sc_x,m2st_sc_y,m2st_sc_z
0,00001,BP_Ada_C_1,34.0,-6.287881,354.167297,14.391177,-4.068234,13.838668,-0.968621,0.066612,...,337.681453,160.060166,-172.649296,7.383960,0.935750,0.127283,0.328893,-0.999921,-0.002682,-0.012263
1,00002,BP_Ada_C_1,17.0,-80.392527,839.159817,70.426628,10.291927,-70.265796,-0.335014,-0.175880,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00003,BP_Ada_C_1,114.0,-50.585528,716.956094,30.322529,-20.436334,29.535145,-0.863197,0.059908,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00004,BP_Ada_C_1,42.0,-20.623707,357.440408,61.804887,-68.575333,9.502579,-0.472476,0.805154,...,386.860524,16.545744,-17.400630,3.438862,-0.960390,0.272383,0.058802,-0.999872,-0.002752,-0.015739
4,00005,BP_Ada_C_1,114.0,-16.400097,540.643029,72.647749,82.989366,63.720576,-0.298245,-0.170291,...,483.475293,165.921985,175.796565,-18.033845,0.967381,-0.071326,0.243077,-0.999933,-0.001848,-0.011442
5,00006,BP_Ada_C_1,92.0,-67.072543,511.871710,35.985887,-115.477473,3.051334,-0.809162,0.306925,...,452.631735,175.191351,-172.912755,-127.839991,0.994123,0.070675,0.081999,-0.999674,-0.011433,-0.022845


In [5]:
# Model Pipeline Setup

base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.PoseLandmarkerOptions(base_options=base_options)
detector = vision.PoseLandmarker.create_from_options(options)

In [6]:
def run_model_pipeline(sample_id, camera_pitch_rad):

    # Load JSON Metadata
    json_path = F"{METADATA_PATH}/{sample_id}A.json"
    with open(json_path, 'r') as file:
        raw_data = json.load(file)

    data = {
        'RS': np.array([raw_data['Right Shoulder Coords'][axis] for axis in ['x', 'y', 'z']]),
        'LS': np.array([raw_data['Left Shoulder Coords'][axis] for axis in ['x', 'y', 'z']]),
        'RT': np.array([raw_data['Right Thigh Coords'][axis] for axis in ['x', 'y', 'z']]),
        'LT': np.array([raw_data['Left Thigh Coords'][axis] for axis in ['x', 'y', 'z']]),
        'RW': np.array([raw_data['Right Wrist Coords'][axis] for axis in ['x', 'y', 'z']]),
    }

    # Get Ground Truth Torso Measurements
    shoulder_midpoint = (data['RS'] + data['LS']) / 2
    thigh_midpoint = (data['RT'] + data['LT']) / 2

    shoulder_width = np.linalg.norm(data['RS'] - data['LS'])
    thigh_width = np.linalg.norm(data['RT'] - data['LT'])
    arm_length = np.linalg.norm(data['RS'] - data['RW'])
    torso_height = np.linalg.norm(shoulder_midpoint - thigh_midpoint)

    # Load Image
    image_A_path = F"{IMAGES_A_PATH}/{sample_id}A.png"
    image = cv2.imread(image_A_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    start_time = time.perf_counter()
    # ------------------------------------

    # Detect Keypoints
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image)
    detection_result = detector.detect(mp_image)

    # Create Numpy Arrays
    landmarks_2d = detection_result.pose_landmarks
    keypoints_2d = np.array([[lm.x, lm.y] for lm in landmarks_2d[0]]) if landmarks_2d else np.zeros((33, 2))

    landmarks_3d = detection_result.pose_world_landmarks
    keypoints_3d = np.array([[lm.x, lm.y, lm.z] for lm in landmarks_3d[0]]) if landmarks_3d else np.zeros((33, 3))

    # Get Predicted Torso Measurements
    L_SHOULDER, R_SHOULDER = 11, 12
    L_HIP, R_HIP = 23, 24
    R_WRIST = 16

    model_shoulder_dist = np.linalg.norm(keypoints_3d[L_SHOULDER] - keypoints_3d[R_SHOULDER])*100
    model_arm_length = np.linalg.norm(keypoints_3d[R_WRIST] - keypoints_3d[R_SHOULDER])*100

    model_shoulder_mid = (keypoints_3d[L_SHOULDER] + keypoints_3d[R_SHOULDER]) / 2
    model_hip_mid = (keypoints_3d[L_HIP] + keypoints_3d[R_HIP]) / 2
    model_torso_height = np.linalg.norm(model_shoulder_mid - model_hip_mid)*100

    # Compute Scale Factor
    ratios = []
    if model_shoulder_dist > 0: ratios.append(shoulder_width / model_shoulder_dist)
    if model_arm_length > 0: ratios.append(arm_length / model_arm_length)
    if model_torso_height > 0: ratios.append(torso_height / model_torso_height)

    if not ratios:
        return [np.nan] * 10, 0

    scale_factor = np.median(ratios)

    # Scale 3D Keypoints
    keypoints_3d_cm = keypoints_3d * 100 * scale_factor

    # Calculate Pixel Focal Length
    focal_length_px = (FOCAL_LENGTH_MM * IMAGE_WIDTH_PX) / SENSOR_WIDTH_MM

    # Get Pixel Distances
    sh_mid_2d = (keypoints_2d[L_SHOULDER] + keypoints_2d[R_SHOULDER]) / 2 * [IMAGE_WIDTH_PX, IMAGE_HEIGHT_PX]
    hi_mid_2d = (keypoints_2d[L_HIP] + keypoints_2d[R_HIP]) / 2 * [IMAGE_WIDTH_PX, IMAGE_HEIGHT_PX]

    def get_pix_dist(idx1, idx2):
        p1 = keypoints_2d[idx1] * [IMAGE_WIDTH_PX, IMAGE_HEIGHT_PX]
        p2 = keypoints_2d[idx2] * [IMAGE_WIDTH_PX, IMAGE_HEIGHT_PX]
        return np.linalg.norm(p1 - p2)

    pixels_shoulders = get_pix_dist(L_SHOULDER, R_SHOULDER)
    pixels_arm = get_pix_dist(R_SHOULDER, R_WRIST)
    pixels_torso = np.linalg.norm(sh_mid_2d - hi_mid_2d)

    # Calculate Distances
    z_estimates = [
        (np.linalg.norm(keypoints_3d_cm[L_SHOULDER] - keypoints_3d_cm[R_SHOULDER]) * focal_length_px) / pixels_shoulders,
        (np.linalg.norm(keypoints_3d_cm[R_SHOULDER] - keypoints_3d_cm[R_WRIST]) * focal_length_px) / pixels_arm,
        (np.linalg.norm(model_shoulder_mid*100*scale_factor - model_hip_mid*100*scale_factor) * focal_length_px) / pixels_torso
    ]

    final_z_distance_cm = np.median(z_estimates)

    hip_center_2d = (keypoints_2d[L_HIP] + keypoints_2d[R_HIP]) / 2
    pixel_x_offset = (hip_center_2d[0] * IMAGE_WIDTH_PX) - (IMAGE_WIDTH_PX / 2)
    pixel_y_offset = (hip_center_2d[1] * IMAGE_HEIGHT_PX) - (IMAGE_HEIGHT_PX / 2)

    final_x_distance_cm = (pixel_x_offset * final_z_distance_cm) / focal_length_px
    final_y_distance_cm = (pixel_y_offset * final_z_distance_cm) / focal_length_px

    # Transform To Camera Coordinates
    transformed_keypoints_3d = keypoints_3d_cm.copy()
    transformed_keypoints_3d[:, 0] += final_x_distance_cm
    transformed_keypoints_3d[:, 1] += final_y_distance_cm
    transformed_keypoints_3d[:, 2] += final_z_distance_cm
    transformed_keypoints_3d /= 100

    # ------------------------------------
    inference_time = time.perf_counter() - start_time

    # Apply Coordinate Conversion
    def convert_to_unreal_coords(points_3d_meters):
        unreal_pts = np.zeros_like(points_3d_meters)
        unreal_pts[:, 0] = points_3d_meters[:, 2] * 100
        unreal_pts[:, 1] = points_3d_meters[:, 0] * 100
        unreal_pts[:, 2] = -points_3d_meters[:, 1] * 100
        return unreal_pts

    ue_keypoints_3d = convert_to_unreal_coords(transformed_keypoints_3d)

    # Given Constants
    camera_coords = np.array([0, 0, 0])
    world_up = np.array([0, 0, 1])

    # Get Right Shoulder & Wrist Coordinates
    shoulder_coords = ue_keypoints_3d[12]
    wrist_coords = ue_keypoints_3d[16]

    # Shoulder-Camera Distance
    distance = np.linalg.norm(shoulder_coords)

    # Shoulder-Wrist Shoulder-Camera Vectors
    shoulder_wrist = wrist_coords - shoulder_coords
    shoulder_wrist /= np.linalg.norm(shoulder_wrist)

    shoulder_camera = camera_coords - shoulder_coords
    shoulder_camera /= np.linalg.norm(shoulder_camera)

    # Angular Separation
    angular_separation_rad = np.arccos(np.clip(np.dot(shoulder_wrist, shoulder_camera), -1.0, 1.0))
    angular_separation_deg = np.rad2deg(angular_separation_rad)

    # Camera Un-Pitch Matrix (Aligns Z-Axis with Gravity)
    c, s = np.cos(-camera_pitch_rad), np.sin(-camera_pitch_rad)
    unpitch_matrix = np.array([
        [c,  0, s],
        [0,  1, 0],
        [-s, 0, c]
    ])

    # Un-Pitch Vectors
    shoulder_wrist_gravity = unpitch_matrix @ shoulder_wrist
    shoulder_wrist_gravity /= np.linalg.norm(shoulder_wrist_gravity)

    shoulder_camera_gravity = unpitch_matrix @ shoulder_camera
    shoulder_camera_gravity /= np.linalg.norm(shoulder_camera_gravity)

    # Yaw & Pitch Components
    shoulder_wrist_gravity_yaw = np.rad2deg(np.atan2(shoulder_wrist_gravity[1], shoulder_wrist_gravity[0]))
    shoulder_wrist_gravity_pitch = np.rad2deg(np.asin(np.clip(shoulder_wrist_gravity[-1], -1.0, 1.0)))

    shoulder_camera_gravity_yaw = np.rad2deg(np.atan2(shoulder_camera_gravity[1], shoulder_camera_gravity[0]))
    shoulder_camera_gravity_pitch = np.rad2deg(np.asin(np.clip(shoulder_camera_gravity[-1], -1.0, 1.0)))

    delta_yaw = (shoulder_wrist_gravity_yaw - shoulder_camera_gravity_yaw + 180) % 360 - 180
    delta_pitch = shoulder_wrist_gravity_pitch - shoulder_camera_gravity_pitch

    # Relevant Outputs
    return distance, angular_separation_deg, delta_yaw, delta_pitch, \
           shoulder_wrist[0], shoulder_wrist[1], shoulder_wrist[2], \
           shoulder_camera[0], shoulder_camera[1], shoulder_camera[2], \
           inference_time

In [7]:
# Execution Loop

active_pipeline = 'm3ad' 
pipeline_cols = [f'{active_pipeline}_{m}' for m in ['dist', 'ang_sep', 'd_yaw', 'd_pitch', 'sw_x', 'sw_y', 'sw_z', 'sc_x', 'sc_y', 'sc_z']]

total_inference_seconds = 0.0
successful_counts = 0

print(f"Running Inference for Pipeline: {active_pipeline}")

for i in tqdm(range(len(df)), desc=f"Processing {active_pipeline}"):
    row_id = df.at[i, 'id']
    
    pitch_deg = df.at[i, 'cam_pitch']
    pitch_rad = np.deg2rad(pitch_deg)
    
    try:
        *results, elapsed = run_model_pipeline(row_id, pitch_rad)
        
        df.loc[i, pipeline_cols] = results
        
        total_inference_seconds += elapsed
        successful_counts += 1

    except Exception as e:
        # tqdm.write(f"Pipeline {active_pipeline} failed for ID {row_id}: {e}")
        continue

print(f"\nTotal Cumulative Inference Time: {total_inference_seconds:.4f} seconds")
if successful_counts > 0:
    print(f"Average per sample: {(total_inference_seconds / successful_counts) * 1000:.2f} ms")

Running Inference for Pipeline: m3ad


Processing m3ad: 100%|██████████| 8192/8192 [08:37<00:00, 15.83it/s]


Total Cumulative Inference Time: 95.2525 seconds
Average per sample: 20.64 ms


In [10]:
df.head(10)

,id,actor,pose,cam_pitch,gt_dist,gt_ang_sep,gt_d_yaw,gt_d_pitch,gt_sw_x,gt_sw_y,...,m2st_dist,m2st_ang_sep,m2st_d_yaw,m2st_d_pitch,m2st_sw_x,m2st_sw_y,m2st_sw_z,m2st_sc_x,m2st_sc_y,m2st_sc_z
0,00001,BP_Ada_C_1,34.0,-6.287881,354.167297,14.391177,-4.068234,13.838668,-0.968621,0.066612,...,337.681453,160.060166,-172.649296,7.383960,0.935750,0.127283,0.328893,-0.999921,-0.002682,-0.012263
1,00002,BP_Ada_C_1,17.0,-80.392527,839.159817,70.426628,10.291927,-70.265796,-0.335014,-0.175880,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00003,BP_Ada_C_1,114.0,-50.585528,716.956094,30.322529,-20.436334,29.535145,-0.863197,0.059908,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00004,BP_Ada_C_1,42.0,-20.623707,357.440408,61.804887,-68.575333,9.502579,-0.472476,0.805154,...,386.860524,16.545744,-17.400630,3.438862,-0.960390,0.272383,0.058802,-0.999872,-0.002752,-0.015739
4,00005,BP_Ada_C_1,114.0,-16.400097,540.643029,72.647749,82.989366,63.720576,-0.298245,-0.170291,...,483.475293,165.921985,175.796565,-18.033845,0.967381,-0.071326,0.243077,-0.999933,-0.001848,-0.011442
5,00006,BP_Ada_C_1,92.0,-67.072543,511.871710,35.985887,-115.477473,3.051334,-0.809162,0.306925,...,452.631735,175.191351,-172.912755,-127.839991,0.994123,0.070675,0.081999,-0.999674,-0.011433,-0.022845
6,00007,BP_Ada_C_1,115.0,-58.337415,573.534355,30.892326,77.429488,21.783261,-0.858134,-0.167461,...,630.444949,8.157476,8.274880,7.165474,-0.990463,-0.060733,0.123671,-0.999988,-0.000535,-0.004866
7,00008,BP_Ada_C_1,65.0,-83.698078,893.154696,40.795695,-94.703134,-33.572474,-0.757044,0.638948,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,00009,BP_Ada_C_1,4.0,-17.160619,582.529679,90.029656,-90.070230,-17.033701,0.000518,0.999997,...,570.897692,24.931222,-24.471015,-7.284804,-0.905390,0.408844,-0.114521,-0.999902,-0.000218,-0.013990
9,00010,BP_Ada_C_1,63.0,-12.268837,727.370530,37.593015,28.693427,27.857188,-0.792364,-0.367116,...,616.070801,173.441593,174.779712,-19.779935,0.993352,-0.086310,0.076174,-0.999975,-0.003789,-0.005986


In [11]:
# Save Dataframe

df.to_csv(SAVE_PATH, index=False)

In [12]:
# Sanity Check

check_df = pd.read_csv(SAVE_PATH, dtype={'id': str})
print(check_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8192 entries, 0 to 8191
Data columns (total 84 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            8192 non-null   object 
 1   actor         8192 non-null   object 
 2   pose          8192 non-null   float64
 3   cam_pitch     8192 non-null   float64
 4   gt_dist       8192 non-null   float64
 5   gt_ang_sep    8192 non-null   float64
 6   gt_d_yaw      8192 non-null   float64
 7   gt_d_pitch    8192 non-null   float64
 8   gt_sw_x       8192 non-null   float64
 9   gt_sw_y       8192 non-null   float64
 10  gt_sw_z       8192 non-null   float64
 11  gt_sc_x       8192 non-null   float64
 12  gt_sc_y       8192 non-null   float64
 13  gt_sc_z       8192 non-null   float64
 14  b3ad_dist     0 non-null      float64
 15  b3ad_ang_sep  0 non-null      float64
 16  b3ad_d_yaw    0 non-null      float64
 17  b3ad_d_pitch  0 non-null      float64
 18  b3ad_sw_x     0 non-null    